# AgroCheck — Treino final (milho + batata)

**Antes de começar:**
1. O `dataset.zip` **ou** a pasta `dataset` (com as 7 pastas de milho e batata) precisa estar no seu Google Drive, em **Meu Drive** (fora de outras pastas). O caderno funciona com qualquer um dos dois.
2. Ligue a placa de vídeo: menu **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4 → Salvar**.
3. Rode as células **uma por vez, de cima pra baixo** (botão ▶ de cada uma), esperando cada uma terminar.

**O que muda em relação ao treino anterior:**
- usa as 7 pastas de folhas de milho e batata, separadas em treino (70%), validação (15%) e teste (15%), sem repetir foto entre elas;
- treina em **duas etapas**: primeiro só a parte final do modelo (rápido), depois um ajuste fino longo e cuidadoso, liberando mais camadas da rede;
- reduz a velocidade de aprendizado automaticamente quando a melhora estaciona;
- para sozinho quando parar de melhorar e guarda a melhor versão;
- o resultado final é medido em fotos que o modelo **nunca viu** (conjunto de teste).

O arquivo gerado se chama `agrocheck_turbinado.keras` e tem **7 classes**. O site precisa ser atualizado para essas 7 classes (a lista certa aparece no final deste caderno).

⏱️ O treino completo pode levar **40 a 70 minutos**. Deixe esta aba aberta e volte nela de vez em quando, senão o Colab pode desconectar por inatividade.

## 1. Conferir a GPU e as versões

In [ ]:
import tensorflow as tf
import keras

print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU ligada:", len(gpus) > 0)
if not gpus:
    print("⚠️ Sem GPU! Vá em Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4.")

## 2. Conectar o Google Drive e separar as 7 pastas do zip
Vai pedir permissão pra acessar o seu Drive: aceite. Essa etapa leva alguns minutos.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import zipfile, os, io, re, random, shutil
from PIL import Image

CAMINHO_ZIP = "/content/drive/MyDrive/dataset.zip"     # se você subiu o arquivo .zip
CAMINHO_PASTA = "/content/drive/MyDrive/dataset"       # se você subiu a pasta aberta
DESTINO = "/content/dados"

# Ordem das classes (o site precisa usar EXATAMENTE esta ordem)
CLASSES = [
    "Corn___Common_Rust_Leaf",
    "Corn___Gray_Leaf_Spot_Leaf",
    "Corn___Healthy_Leaf",
    "Corn___Northern_Leaf_Blight_Leaf",
    "Potato___Early_blight_Leaf",
    "Potato___Late_Blight_Leaf",
    "Potato__Healthy_Leaf",
]
NOMES_PT = [
    "Milho - Ferrugem comum",
    "Milho - Mancha cinzenta",
    "Milho saudável",
    "Milho - Helmintosporiose",
    "Batata - Pinta-preta",
    "Batata - Requeima",
    "Batata saudável",
]
EXTENSOES = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

def normalizar(nome):
    # ignora diferença de maiúsculas e de quantidade de "_" no nome das pastas
    return re.sub(r"_+", "_", nome).lower()

def eh_da_classe(caminho, classe):
    partes = caminho.replace("\\", "/").split("/")[:-1]
    return normalizar(classe) in [normalizar(p) for p in partes]

# Descobre se vamos ler do .zip ou da pasta
z = None
if os.path.exists(CAMINHO_ZIP):
    print("Lendo do arquivo dataset.zip...")
    z = zipfile.ZipFile(CAMINHO_ZIP)
    todos = [n for n in z.namelist() if n.lower().endswith(EXTENSOES)]
    ler = lambda caminho: z.read(caminho)
elif os.path.isdir(CAMINHO_PASTA):
    print("Lendo da pasta dataset do Drive (pode levar alguns minutos)...")
    todos = []
    for raiz, _, arquivos in os.walk(CAMINHO_PASTA):
        for a in arquivos:
            if a.lower().endswith(EXTENSOES):
                todos.append(os.path.join(raiz, a))
    def ler(caminho):
        with open(caminho, "rb") as f:
            return f.read()
else:
    raise FileNotFoundError("Não achei nem dataset.zip nem a pasta dataset em Meu Drive.")

if os.path.exists(DESTINO):
    shutil.rmtree(DESTINO)

random.seed(42)
for classe in CLASSES:
    arquivos = sorted(n for n in todos if eh_da_classe(n, classe))
    if not arquivos:
        raise ValueError(f"Não encontrei fotos da pasta {classe}.")

    random.shuffle(arquivos)
    n = len(arquivos)
    n_treino = int(n * 0.70)
    n_val = int(n * 0.15)
    partes = {
        "treino": arquivos[:n_treino],
        "validacao": arquivos[n_treino:n_treino + n_val],
        "teste": arquivos[n_treino + n_val:],
    }

    defeituosas = 0
    for parte, lista in partes.items():
        pasta = os.path.join(DESTINO, parte, classe)
        os.makedirs(pasta, exist_ok=True)
        for i, caminho in enumerate(lista):
            try:
                # Regrava toda foto como JPG colorido (RGB) para evitar erro de formato
                with Image.open(io.BytesIO(ler(caminho))) as img:
                    img.convert("RGB").save(os.path.join(pasta, f"{i:05d}.jpg"), quality=95)
            except Exception:
                defeituosas += 1

    print(f"{classe}: {n} fotos → treino {len(partes['treino'])}, "
          f"validação {len(partes['validacao'])}, teste {len(partes['teste'])}"
          + (f" ({defeituosas} com defeito ignoradas)" if defeituosas else ""))

print("\nSeparação concluída! 🌱")

## 3. Preparar os dados para o TensorFlow

In [ ]:
import numpy as np

TAMANHO = (224, 224)
LOTE = 32

def carregar(parte, embaralhar):
    return tf.keras.utils.image_dataset_from_directory(
        os.path.join(DESTINO, parte),
        class_names=CLASSES,        # garante a ordem das classes
        image_size=TAMANHO,
        batch_size=LOTE,
        shuffle=embaralhar,
        seed=42,
    )

treino = carregar("treino", True).prefetch(tf.data.AUTOTUNE)
validacao = carregar("validacao", False).prefetch(tf.data.AUTOTUNE)
teste = carregar("teste", False).prefetch(tf.data.AUTOTUNE)

# Pesos para compensar classes com menos fotos
contagem = np.array([len(os.listdir(os.path.join(DESTINO, "treino", c))) for c in CLASSES])
pesos = {i: float(contagem.sum() / (len(CLASSES) * qtd)) for i, qtd in enumerate(contagem)}
for nome, qtd, i in zip(NOMES_PT, contagem, range(len(CLASSES))):
    print(f"{nome}: {qtd} fotos de treino (peso {pesos[i]:.2f})")

## 4. Montar o modelo (MobileNetV2 + transferência de aprendizado)

In [ ]:
from tensorflow.keras import layers

aumento_de_dados = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.1),
], name="aumento_de_dados")

base = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3), include_top=False, weights="imagenet"
)
base.trainable = False  # etapa 1: base congelada

entrada = tf.keras.Input(shape=(224, 224, 3))
x = aumento_de_dados(entrada)
x = layers.Rescaling(1./127.5, offset=-1)(x)   # o site NÃO divide por 255, então isso fica aqui dentro
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
saida = layers.Dense(len(CLASSES), activation="softmax")(x)

modelo = tf.keras.Model(entrada, saida, name="agrocheck")
modelo.summary()

## 5. Etapa 1 — treinar só a parte final (rápido)
A base MobileNetV2 fica congelada. Para sozinho quando parar de melhorar.

In [ ]:
modelo.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

historico1 = modelo.fit(
    treino,
    validation_data=validacao,
    epochs=20,
    class_weight=pesos,
    callbacks=[tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=4, restore_best_weights=True
    )],
)
print("Melhor acurácia de validação na etapa 1:", max(historico1.history["val_accuracy"]))

## 6. Etapa 2 — ajuste fino (fine-tuning), longo e cuidadoso
Libera cerca de 1/3 final da MobileNetV2 para se adaptar às folhas, com aprendizado bem lento. Quando a melhora estaciona, ele diminui a velocidade sozinho; quando para de melhorar de vez, encerra e guarda a melhor versão. **Esta é a etapa mais demorada.**

In [ ]:
base.trainable = True
for camada in base.layers[:100]:   # congela as 100 primeiras, libera as ~54 finais
    camada.trainable = False
for camada in base.layers:
    if isinstance(camada, layers.BatchNormalization):
        camada.trainable = False   # deixa as normalizações fixas (mais estável)

modelo.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

epocas_feitas = len(historico1.history["loss"])
historico2 = modelo.fit(
    treino,
    validation_data=validacao,
    epochs=epocas_feitas + 40,
    initial_epoch=epocas_feitas,
    class_weight=pesos,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=7, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=3,
                                             min_lr=1e-7, verbose=1),
        tf.keras.callbacks.ModelCheckpoint("agrocheck_turbinado.keras", monitor="val_loss",
                                           save_best_only=True, verbose=1),
    ],
)
print("Melhor acurácia de validação na etapa 2:", max(historico2.history["val_accuracy"]))

## 7. Gráficos da evolução do treino

In [ ]:
import matplotlib.pyplot as plt

acc = historico1.history["accuracy"] + historico2.history["accuracy"]
val_acc = historico1.history["val_accuracy"] + historico2.history["val_accuracy"]
perda = historico1.history["loss"] + historico2.history["loss"]
val_perda = historico1.history["val_loss"] + historico2.history["val_loss"]
epocas = range(1, len(acc) + 1)
inicio_ft = len(historico1.history["loss"]) + 0.5

plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(epocas, acc, "o-", label="Treino")
plt.plot(epocas, val_acc, "o-", label="Validação")
plt.axvline(inicio_ft, color="gray", linestyle="--", label="Início do ajuste fino")
plt.title("Acurácia"); plt.xlabel("Época"); plt.legend(); plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(epocas, perda, "o-", label="Treino")
plt.plot(epocas, val_perda, "o-", label="Validação")
plt.axvline(inicio_ft, color="gray", linestyle="--", label="Início do ajuste fino")
plt.title("Perda (loss)"); plt.xlabel("Época"); plt.legend(); plt.grid(True)
plt.tight_layout(); plt.show()

## 8. Resultado final no conjunto de TESTE (fotos que o modelo nunca viu)
Esse é o número honesto para colocar na apresentação.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

perda_teste, acc_teste = modelo.evaluate(teste, verbose=0)
print(f"🎯 Acurácia no teste: {acc_teste*100:.2f}%\n")

y_real = np.concatenate([y.numpy() for _, y in teste])
y_prev = np.argmax(modelo.predict(teste, verbose=0), axis=1)

print(classification_report(y_real, y_prev, target_names=NOMES_PT, digits=4))

plt.figure(figsize=(10, 8))
sns.heatmap(confusion_matrix(y_real, y_prev), annot=True, fmt="d", cmap="Greens",
            xticklabels=NOMES_PT, yticklabels=NOMES_PT)
plt.title("Matriz de confusão — conjunto de teste")
plt.xlabel("Previsão da IA"); plt.ylabel("Classe real")
plt.xticks(rotation=45, ha="right"); plt.tight_layout(); plt.show()

## 9. Salvar e baixar o modelo
Salva uma cópia no seu Drive e baixa para o computador. Depois é só colocar o arquivo na pasta do site, substituindo o antigo.

In [ ]:
from google.colab import files

modelo.save("agrocheck_turbinado.keras")

# Confere se o arquivo salvo abre e tem 6 saídas
conferencia = tf.keras.models.load_model("agrocheck_turbinado.keras")
print("Saídas do modelo:", conferencia.output_shape[-1], "(tem que ser 7)")

shutil.copy("agrocheck_turbinado.keras", "/content/drive/MyDrive/agrocheck_turbinado.keras")

print("\nLista de classes para o site (nesta ordem):")
for i, nome in enumerate(NOMES_PT):
    print(f"  {i}: {nome}")
print("Cópia salva em Meu Drive ✅")

files.download("agrocheck_turbinado.keras")